# Nettoyage des données

Cette section vise à préparer les données brutes avant toute analyse.
Nous appliquons un ensemble de règles pour corriger les incohérences et harmoniser les variables :

- Suppression des doublons d’identifiants.

- Suppression des années aberrantes (1893, 1899, 1900).

- Nettoyage des valeurs du statut marital et correction des valeurs absurdes (“Absurd”, “YOLO”).

- Remplacement des valeurs extrêmes de revenu (666666) par la médiane.

- Conversion automatique des colonnes en types numériques ou catégoriels.

- Imputation des valeurs manquantes par médiane (numériques) ou mode (catégorielles).

- Transformation des variables binaires textuelles (yes / no, etc.) en 0/1.

- Création d’un schéma de données unifié en français (to_french_schema) garantissant zéro valeur manquante.

- Enfin, un rapport automatique affiche un résumé du nettoyage (valeurs supprimées, médianes utilisées, etc.).

In [ ]:
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype, is_numeric_dtype

# Constantes
POS = {
    "yes","oui","true","t","1","accepted","converted",
    "purchase","purchased","buy","bought","success","y"
}

FR_ORDER = [
    "Identifiant","Année_naissance","Education","Situation_matrimoniale","Revenu",
    "Enfant_charge","Ado_charge","Date_acquisition_client","Nombre_jours_depuis_dernier_achat",
    "Montant_vin","Montant_fruits","Montant_viande","Montant_poisson","Montant_sucreries","Montant_luxe",
    "Nb_achats_promo","Nb_achats_en_ligne","Achats_catalogue","Achats_magasin",
    "Nb_visites_web_mois",
    "Accepte_Campagne_3","Accepte_Campagne_4","Accepte_Campagne_5","Accepte_Campagne_1","Accepte_Campagne_2",
    "Reclamation_client","Z_CostContact","Z_Revenue","Response"
]

# Utils
def bar(title: str, ch: str="═", width: int=70):
    t = f" {title} "; n = max(0, width - len(t)); L = n // 2; R = n - L
    return f"{ch*L}{t}{ch*R}"

def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip().lower().replace(" ", "_") for c in out.columns]
    return out

def parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if "date" in c or c.startswith("dt_"):
            out[c] = pd.to_datetime(out[c], errors="coerce", dayfirst=True)
    return out

def to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def to_int01(s) -> pd.Series:
    sr = pd.to_numeric(s, errors="coerce").fillna(0).astype(int)
    return sr.clip(lower=0, upper=1)

# Nettoyage (règles métier)
def clean_business_rules(df: pd.DataFrame):
    log = {}
    df = normalize_cols(df)

    # trim objets
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()

    # dates
    df = parse_dates(df)

    # (1) unicité id
    id_col = next((c for c in ["id","client_id","customer_id"] if c in df.columns), None)
    if id_col:
        before = len(df)
        df = df.drop_duplicates(subset=[id_col])
        log["dup_id_removed"] = before - len(df)

    # (2) années aberrantes
    yb_col = next((c for c in ["year_birth","birth_year","annee_naissance","year_of_birth"] if c in df.columns), None)
    if yb_col:
        df[yb_col] = to_num(df[yb_col])
        bad = [1893, 1899, 1900]
        rm = int(df[yb_col].isin(bad).sum())
        df = df.loc[~df[yb_col].isin(bad)].copy()
        log["bad_years_removed"] = rm

    # (3) statut marital (Alone→Single) + drop Absurd/YOLO
    m_col = next((c for c in ["marital_status","status_marital","marital","statut_marital"] if c in df.columns), None)
    if m_col:
        df[m_col] = df[m_col].astype(str).str.strip().str.lower()
        MAP = {
            "alone": "single", "single": "single",
            "married": "married", "together": "together",
            "divorced": "divorced", "widow": "widow", "widowed": "widow",
        }
        BAD = {"absurd","yolo"}
        before = len(df)
        df = df.loc[~df[m_col].isin(BAD)].copy()
        log["marital_bad_rows_removed"] = before - len(df)
        df[m_col] = df[m_col].map(MAP).fillna(df[m_col])
        df[m_col] = df[m_col].str.title()
        log["marital_modalities"] = sorted(df[m_col].dropna().unique().tolist())

    # (4) income: 666666 -> NA -> médiane
    if "income" in df.columns:
        df["income"] = to_num(df["income"])
        outliers = int((df["income"] == 666_666).sum())
        if outliers:
            df.loc[df["income"] == 666_666, "income"] = np.nan
        med = float(df["income"].median(skipna=True))
        df["income"] = df["income"].fillna(med)
        log["income_outliers_666666"] = outliers
        log["income_median_used"] = med

    # (5) conversion large vers numérique (coerce)
    keep_cat = {m_col or "", "education", "response"}
    for c in df.columns:
        if c not in keep_cat:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # (6) imputation simple
    for c in df.columns:
        if is_numeric_dtype(df[c]):
            df[c] = df[c].fillna(df[c].median())
        else:
            mode = df[c].mode(dropna=True)
            df[c] = df[c].fillna(mode.iloc[0] if not mode.empty else "Unknown")

    # (7) binaires texte -> 0/1 (exclure tous les compteurs pour éviter la binarisation accidentelle)
    EXCLUDE_BIN = {
        "kidhome", "teenhome",                           # enfants/ados à charge (comptes)
        "numdealspurchases", "numwebpurchases",
        "numcatalogpurchases", "numstorepurchases",
        "numwebvisitsmonth", "recency",
        "mntwines", "mntfruits", "mntmeatproducts",
        "mntfishproducts", "mntsweetproducts", "mntgoldprods",
        "income", "z_costcontact", "z_revenue"
    }
    for c in df.columns:
        if (c not in EXCLUDE_BIN) and (not is_numeric_dtype(df[c])) and df[c].nunique(dropna=True) == 2:
            df[c] = df[c].astype(str).str.lower().map(lambda v: 1 if v in POS else 0)

    # (8) doublons globaux
    df = df.drop_duplicates()

    return df, log

# Schéma FR + zéro NA
def to_french_schema(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "Identifiant":                  to_num(df.get("id")).fillna(0).astype(int),
        "Année_naissance":              to_num(df.get("year_birth")).astype("Int64"),
        "Education":                    df.get("education").astype(str),
        "Situation_matrimoniale":       df.get("marital_status").astype(str),
        "Revenu":                       to_num(df.get("income")).fillna(0).astype(float),
        "Enfant_charge":                to_num(df.get("kidhome")).fillna(0).astype(int),   # ← nombre
        "Ado_charge":                   to_num(df.get("teenhome")).fillna(0).astype(int),  # ← nombre
        "Date_acquisition_client":      pd.to_datetime(df.get("dt_customer"), errors="coerce"),
        "Nombre_jours_depuis_dernier_achat":
                                        to_num(df.get("recency")).fillna(0).astype(int),
        "Montant_vin":                  to_num(df.get("mntwines")).fillna(0).astype(float),
        "Montant_fruits":               to_num(df.get("mntfruits")).fillna(0).astype(float),
        "Montant_viande":               to_num(df.get("mntmeatproducts")).fillna(0).astype(float),
        "Montant_poisson":              to_num(df.get("mntfishproducts")).fillna(0).astype(float),
        "Montant_sucreries":            to_num(df.get("mntsweetproducts")).fillna(0).astype(float),
        "Montant_luxe":                 to_num(df.get("mntgoldprods")).fillna(0).astype(float),
        "Nb_achats_promo":              to_num(df.get("numdealspurchases")).fillna(0).astype(int),
        "Nb_achats_en_ligne":           to_num(df.get("numwebpurchases")).fillna(0).astype(int),
        "Achats_catalogue":             to_num(df.get("numcatalogpurchases")).fillna(0).astype(int),
        "Achats_magasin":               to_num(df.get("numstorepurchases")).fillna(0).astype(int),
        "Nb_visites_web_mois":          to_num(df.get("numwebvisitsmonth")).fillna(0).astype(int),
        "Accepte_Campagne_3":           to_int01(df.get("acceptedcmp3", 0)),
        "Accepte_Campagne_4":           to_int01(df.get("acceptedcmp4", 0)),
        "Accepte_Campagne_5":           to_int01(df.get("acceptedcmp5", 0)),
        "Accepte_Campagne_1":           to_int01(df.get("acceptedcmp1", 0)),
        "Accepte_Campagne_2":           to_int01(df.get("acceptedcmp2", 0)),
        "Reclamation_client":           to_num(df.get("complain")).fillna(0).astype(int),
        "Z_CostContact":                to_num(df.get("z_costcontact")).fillna(0).astype(float),
        "Z_Revenue":                    to_num(df.get("z_revenue")).fillna(0).astype(float),
        "Response":                     to_int01(df.get("response", 0)),
    })

    # ordre strict
    out = out[FR_ORDER]

    # zéro NA garanti
    for c in out.columns:
        if is_datetime64_any_dtype(out[c]):
            out[c] = out[c].fillna(pd.Timestamp("1970-01-01"))
        elif is_numeric_dtype(out[c]):
            out[c] = out[c].fillna(0)
        else:
            out[c] = out[c].fillna("Unknown")

    assert not out.isna().any().any(), "Il reste des NA dans le schéma FR."
    return out

# Rapport
def quick_report(df: pd.DataFrame, log: dict, src: str, dst: Path):
    print(bar("RAPPORT DE NETTOYAGE", "█"))
    print(f"Source : {src}")
    print(f"Sortie : {dst}\n")

    print(bar("ACTIONS EFFECTUÉES"))
    print(f"• Doublons supprimés sur id           : {log.get('dup_id_removed', 0)}")
    print(f"• Années aberrantes retirées          : {log.get('bad_years_removed', 0)} (1893, 1899, 1900)")
    print(f"• Lignes marital absurdes retirées    : {log.get('marital_bad_rows_removed', 0)} (Absurd/YOLO)")
    imed = log.get("income_median_used", float("nan"))
    print(f"• Income=666666 corrigés              : {log.get('income_outliers_666666', 0)}")
    print(f"• Médiane income utilisée             : {imed:.0f}" if pd.notna(imed) else "• Médiane income utilisée             : —")
    mods = log.get("marital_modalities", [])
    print(f"• Modalités marital finales           : {', '.join(mods) if mods else '—'}")

    print("\n" + bar("VUE D’ENSEMBLE"))
    num_cols = [c for c in df.columns if is_numeric_dtype(df[c])]
    cat_cols = [c for c in df.columns if not is_numeric_dtype(df[c])]
    print(f"• Lignes x Colonnes                   : {df.shape[0]} x {df.shape[1]}")
    print(f"• Numériques / Catégorielles          : {len(num_cols)} / {len(cat_cols)}")
    print(f"• Zéro NA garanti                     : {not df.isna().any().any()}")

    if "Année_naissance" in df.columns:
        y = df["Année_naissance"].astype("Int64").dropna()
        if not y.empty:
            print("\n" + bar("RÉSUMÉ — Année_naissance"))
            print({"min": int(y.min()), "p50": int(y.median()), "max": int(y.max())})

    if "Revenu" in df.columns:
        s = df["Revenu"].astype(float)
        if not s.empty:
            print("\n" + bar("RÉSUMÉ — Revenu"))
            print({"min": float(s.min()), "p50": float(s.median()), "p95": float(np.percentile(s,95)), "max": float(s.max())})

    if "Nombre_jours_depuis_dernier_achat" in df.columns:
        r = df["Nombre_jours_depuis_dernier_achat"].astype(int)
        if not r.empty:
            print("\n" + bar("RÉSUMÉ — Recency (jours)"))
            print({"min": int(r.min()), "p50": int(np.median(r)), "p95": int(np.percentile(r,95)), "max": int(r.max())})

    print("\n" + bar("FIN DU RAPPORT", "█"))

# CLI
def main():
    ap = argparse.ArgumentParser(description="Nettoyage marketing → CSV (pas de Parquet)")
    ap.add_argument("--input", required=True, help="Chemin du CSV brut (séparateur ';')")
    ap.add_argument("--out", required=True, help="Chemin de sortie CSV propre (sera créé)")
    ap.add_argument("--sep", default=";", help="Séparateur CSV (défaut: ;)")
    ap.add_argument("--encoding", default="utf-8", help="Encodage (défaut: utf-8)")
    args = ap.parse_args()

    # Lecture
    df_raw = pd.read_csv(args.input, sep=args.sep, encoding=args.encoding)

    # Nettoyage + règles métier
    df_clean, log = clean_business_rules(df_raw)

    # Schéma FR + garantie zéro NA
    df_fr = to_french_schema(df_clean)

    # Écriture CSV
    out_path = Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_fr.to_csv(out_path, index=False)

    # Rapport terminal
    quick_report(df_fr, log, args.input, out_path)

if __name__ == "__main__":
    main()


# Feature Engineering

Dans cette partie, nous enrichissons le jeu de données avec de nouvelles variables dérivées pour mieux décrire le comportement client.
Ces indicateurs facilitent les analyses futures et les modèles de segmentation. Voici les principales variables créées :

- Dépense_totale : somme de toutes les dépenses par client.

- Dépense_plaisir : dépenses liées aux produits de plaisir (vin, sucreries, luxe).

- Part_plaisir : part de la dépense plaisir dans la dépense totale.

- Taux_depense_sur_revenu : ratio des dépenses par rapport au revenu.

- Total_achats : total des achats tous canaux confondus.

- Part_achats_en_ligne / catalogue : répartition des achats par canal.

- Taux_visite_achat_web : taux de conversion web.

- Nb_enfants_total : total des enfants et adolescents à charge.

- Âge : calculé à partir de l’année de naissance (année de référence : 2014).

- Score_engagement_marketing : somme des campagnes marketing acceptées.

- Taux_acceptation_campagne : score d’engagement moyen sur 5 campagnes.

Ces nouvelles features sont ensuite enregistrées dans un CSV enrichi, prêt pour la segmentation.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
features_engineering.py — création d'indicateurs utiles à partir du dataset marketing.

Exemple d'usage :
  python features_engineering.py --input data/camp_market_clean.csv --output processed/features_enriched.csv --ref-year 2014
"""

import argparse
import math
import sys
import numpy as np
import pandas as pd

# ------------------ utils ------------------

def ensure_numeric(df: pd.DataFrame, cols):
    """Force cols en float (gère virgules décimales, espaces, symboles)."""
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c].astype(str)
        s = s.str.replace(",", ".", regex=False)
        s = s.str.replace(r"[^0-9\.\-eE+]", "", regex=True)
        df[c] = pd.to_numeric(s, errors="coerce")
    return df

def safe_div(num, den):
    if den is None or (isinstance(den, (int, float, np.floating)) and (den == 0 or np.isnan(den))):
        return np.nan
    try:
        return num / den
    except Exception:
        return np.nan

# ------------------ mapping colonnes ------------------
# Le script accepte deux schémas :
# - EN (Kaggle/Portugal) : MntWines, MntFruits, ..., NumWebPurchases, Income, etc.
# - FR (ton CSV) : Montant_vin, Montant_fruits, ..., Nb_achats_en_ligne, Revenu, etc.
COLMAPS = [
    # Schéma EN
    {
        "vin": "MntWines",
        "fruits": "MntFruits",
        "viande": "MntMeatProducts",
        "poisson": "MntFishProducts",
        "sucreries": "MntSweetProducts",
        "luxe": "MntGoldProds",
        "achats_web": "NumWebPurchases",
        "achats_catalogue": "NumCatalogPurchases",
        "achats_magasin": "NumStorePurchases",
        "visites_web": "NumWebVisitsMonth",
        "enfant": "Kidhome",
        "ado": "Teenhome",
        "annee_naissance": "Year_Birth",
        "revenu": "Income",
        "cmp1": "AcceptedCmp1",
        "cmp2": "AcceptedCmp2",
        "cmp3": "AcceptedCmp3",
        "cmp4": "AcceptedCmp4",
        "cmp5": "AcceptedCmp5",
    },
    # Schéma FR
    {
        "vin": "Montant_vin",
        "fruits": "Montant_fruits",
        "viande": "Montant_viande",
        "poisson": "Montant_poisson",
        "sucreries": "Montant_sucreries",
        "luxe": "Montant_luxe",
        "achats_web": "Nb_achats_en_ligne",
        "achats_catalogue": "Achats_catalogue",
        "achats_magasin": "Achats_magasin",
        "visites_web": "Nb_visites_web_mois",
        "enfant": "Enfant_charge",
        "ado": "Ado_charge",
        "annee_naissance": "Année_naissance",
        "revenu": "Revenu",
        "cmp1": "Accepte_Campagne_1",
        "cmp2": "Accepte_Campagne_2",
        "cmp3": "Accepte_Campagne_3",
        "cmp4": "Accepte_Campagne_4",
        "cmp5": "Accepte_Campagne_5",
    },
]

def pick_schema(df: pd.DataFrame):
    """Choisit automatiquement le mapping le plus compatible avec le DF."""
    best = None
    best_score = -1
    for m in COLMAPS:
        keys = list(m.values())
        score = sum(1 for k in keys if k in df.columns)
        if score > best_score:
            best = m
            best_score = score
    return best

# ------------------ core ------------------

def engineer_features(df: pd.DataFrame, ref_year: int = 2014) -> pd.DataFrame:
    m = pick_schema(df)
    if m is None:
        raise ValueError("Impossible de détecter le schéma de colonnes (FR/EN).")

    # colonnes nécessaires minimum
    required = [
        m["vin"], m["fruits"], m["viande"], m["poisson"], m["sucreries"], m["luxe"],
        m["achats_web"], m["achats_catalogue"], m["achats_magasin"],
        m["visites_web"], m["enfant"], m["ado"], m["annee_naissance"], m["revenu"],
        m["cmp1"], m["cmp2"], m["cmp3"], m["cmp4"], m["cmp5"]
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Colonnes manquantes : {missing}")

    # force numériques
    num_cols = [
        m["vin"], m["fruits"], m["viande"], m["poisson"], m["sucreries"], m["luxe"],
        m["achats_web"], m["achats_catalogue"], m["achats_magasin"], m["visites_web"],
        m["enfant"], m["ado"], m["annee_naissance"], m["revenu"],
        m["cmp1"], m["cmp2"], m["cmp3"], m["cmp4"], m["cmp5"],
    ]
    df = ensure_numeric(df, num_cols)

    # 1) Dépense_totale
    df["Depense_totale"] = (
        df[m["vin"]]
        + df[m["fruits"]]
        + df[m["viande"]]
        + df[m["poisson"]]
        + df[m["sucreries"]]
        + df[m["luxe"]]
    )

    # 2) Depense_plaisir
    df["Depense_plaisir"] = df[m["vin"]] + df[m["sucreries"]] + df[m["luxe"]]

    # 3) Part_plaisir
    df["Part_plaisir"] = df["Depense_plaisir"] / df["Depense_totale"].replace(0, np.nan)

    # 4) Taux_depense_sur_revenu
    df["Taux_depense_sur_revenu"] = df["Depense_totale"] / df[m["revenu"]].replace(0, np.nan)

    # 5) Total_achats
    df["Total_achats"] = df[m["achats_web"]] + df[m["achats_catalogue"]] + df[m["achats_magasin"]]

    # 6) Part_achats_en_ligne
    df["Part_achats_en_ligne"] = df[m["achats_web"]] / df["Total_achats"].replace(0, np.nan)

    # 7) Part_achats_catalogue
    df["Part_achats_catalogue"] = df[m["achats_catalogue"]] / df["Total_achats"].replace(0, np.nan)

    # 8) Taux_visite_achat_web
    df["Taux_visite_achat_web"] = df[m["achats_web"]] / df[m["visites_web"]].replace(0, np.nan)

    # 9) Nb_enfants_total
    df["Nb_enfants_total"] = df[m["enfant"]] + df[m["ado"]]

    # 10) Age
    df["Age"] = ref_year - df[m["annee_naissance"]]

    # 11) Score_engagement_marketing
    df["Score_engagement_marketing"] = (
        df[m["cmp1"]] + df[m["cmp2"]] + df[m["cmp3"]] + df[m["cmp4"]] + df[m["cmp5"]]
    )

    # 12) Taux_acceptation_campagne
    df["Taux_acceptation_campagne"] = df["Score_engagement_marketing"] / 5.0

    return df

# ------------------ CLI ------------------

def main():
    p = argparse.ArgumentParser(description="Création d'indicateurs (feature engineering)")
    p.add_argument("--input", required=True, help="Chemin du CSV d'entrée")
    p.add_argument("--output", required=True, help="Chemin du CSV enrichi à écrire")
    p.add_argument("--ref-year", type=int, default=2014, help="Année de référence pour le calcul de l'âge (défaut 2014)")
    args = p.parse_args()

    try:
        df = pd.read_csv(args.input)
    except Exception as e:
        print(f"[ERREUR] Lecture CSV: {e}")
        sys.exit(1)

    try:
        df_out = engineer_features(df, ref_year=args.ref_year)
    except Exception as e:
        print(f"[ERREUR] Calcul des features: {e}")
        sys.exit(2)

    try:
        df_out.to_csv(args.output, index=False)
    except Exception as e:
        print(f"[ERREUR] Écriture CSV: {e}")
        sys.exit(3)

    # Rendu terminal concis
    n_rows, n_cols = df_out.shape
    created_cols = [
        "Depense_totale","Depense_plaisir","Part_plaisir","Taux_depense_sur_revenu",
        "Total_achats","Part_achats_en_ligne","Part_achats_catalogue","Taux_visite_achat_web",
        "Nb_enfants_total","Age","Score_engagement_marketing","Taux_acceptation_campagne"
    ]

    print("\n" + "="*72)
    print(" FEATURES CRÉÉES ".center(72, "="))
    print("="*72)
    for c in created_cols:
        miss = df_out[c].isna().mean()
        print(f" - {c:28s} | NaN: {miss:6.2%} | exemple: {df_out[c].dropna().head(1).to_list()[0] if df_out[c].notna().any() else '—'}")
    print("-"*72)
    print(f"Lignes: {n_rows:,}  |  Colonnes: {n_cols:,}".replace(",", " "))
    print(f"Fichier écrit -> {args.output}")
    print("="*72 + "\n")

if __name__ == "__main__":
    main()


# Corrélation de Pearson

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Import du fichier ---
df = pd.read_csv("data/processed/camp_market_clean.csv")

# --- Matrice de corrélation complète ---
corr_matrix = df.corr(method='pearson')

# --- Filtrer uniquement les corrélations fortes entre [-1, -0.3] ou [0.3, 1] ---
strong_corr = corr_matrix[(corr_matrix <= -0.3) | (corr_matrix >= 0.3)]

# --- Affichage avec heatmap ---
plt.figure(figsize=(20, 15))
sns.heatmap(
    strong_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    cbar_kws={'label': 'Coefficient de corrélation'}
)
plt.title("🔥 Corrélations fortes (|r| ≥ 0.3)")
plt.show()

# --- Afficher aussi la liste triée des corrélations fortes ---
corr_pairs = (
    corr_matrix.unstack()
    .reset_index()
    .rename(columns={'level_0': 'Variable_1', 'level_1': 'Variable_2', 0: 'Corrélation'})
)
# Supprimer les doublons (corrélation A-B = B-A)
corr_pairs = corr_pairs[corr_pairs['Variable_1'] < corr_pairs['Variable_2']]
# Garder uniquement les corrélations fortes
corr_pairs = corr_pairs[(corr_pairs['Corrélation'] >= 0.3) | (corr_pairs['Corrélation'] <= -0.3)]
# Trier par valeur absolue décroissante
corr_pairs = corr_pairs.reindex(corr_pairs['Corrélation'].abs().sort_values(ascending=False).index)

display(corr_pairs)

ValueError: could not convert string to float: 'Graduation'

# Segmentation client (Clustering)

L’objectif ici est de segmenter la clientèle selon des comportements d’achat et des caractéristiques socio-économiques similaires.

Cette partie regroupe plusieurs étapes :

- Préparation des données : sélection des variables pertinentes (issues du feature engineering).

- Standardisation des variables pour éviter les biais liés aux échelles différentes.

- Application d’un algorithme de clustering.

- Évaluation des clusters (inertie, silhouette score, etc.).

- Analyse et interprétation des groupes de clients pour comprendre leurs profils (par exemple : "gros dépensiers", "familles nombreuses", "acheteurs en ligne", etc.).

Cette segmentation permet d’identifier des groupes homogènes afin d’orienter les campagnes marketing et d’améliorer la stratégie commerciale.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
features_engineering.py — création d'indicateurs utiles à partir du dataset marketing.

Exemple d'usage :
  python features_engineering.py --input data/camp_market_clean.csv --output processed/features_enriched.csv --ref-year 2014
"""

import argparse
import math
import sys
import numpy as np
import pandas as pd

# ------------------ utils ------------------

def ensure_numeric(df: pd.DataFrame, cols):
    """Force cols en float (gère virgules décimales, espaces, symboles)."""
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c].astype(str)
        s = s.str.replace(",", ".", regex=False)
        s = s.str.replace(r"[^0-9\.\-eE+]", "", regex=True)
        df[c] = pd.to_numeric(s, errors="coerce")
    return df

def safe_div(num, den):
    if den is None or (isinstance(den, (int, float, np.floating)) and (den == 0 or np.isnan(den))):
        return np.nan
    try:
        return num / den
    except Exception:
        return np.nan

# ------------------ mapping colonnes ------------------
# Le script accepte deux schémas :
# - EN (Kaggle/Portugal) : MntWines, MntFruits, ..., NumWebPurchases, Income, etc.
# - FR (ton CSV) : Montant_vin, Montant_fruits, ..., Nb_achats_en_ligne, Revenu, etc.
COLMAPS = [
    # Schéma EN
    {
        "vin": "MntWines",
        "fruits": "MntFruits",
        "viande": "MntMeatProducts",
        "poisson": "MntFishProducts",
        "sucreries": "MntSweetProducts",
        "luxe": "MntGoldProds",
        "achats_web": "NumWebPurchases",
        "achats_catalogue": "NumCatalogPurchases",
        "achats_magasin": "NumStorePurchases",
        "visites_web": "NumWebVisitsMonth",
        "enfant": "Kidhome",
        "ado": "Teenhome",
        "annee_naissance": "Year_Birth",
        "revenu": "Income",
        "cmp1": "AcceptedCmp1",
        "cmp2": "AcceptedCmp2",
        "cmp3": "AcceptedCmp3",
        "cmp4": "AcceptedCmp4",
        "cmp5": "AcceptedCmp5",
    },
    # Schéma FR
    {
        "vin": "Montant_vin",
        "fruits": "Montant_fruits",
        "viande": "Montant_viande",
        "poisson": "Montant_poisson",
        "sucreries": "Montant_sucreries",
        "luxe": "Montant_luxe",
        "achats_web": "Nb_achats_en_ligne",
        "achats_catalogue": "Achats_catalogue",
        "achats_magasin": "Achats_magasin",
        "visites_web": "Nb_visites_web_mois",
        "enfant": "Enfant_charge",
        "ado": "Ado_charge",
        "annee_naissance": "Année_naissance",
        "revenu": "Revenu",
        "cmp1": "Accepte_Campagne_1",
        "cmp2": "Accepte_Campagne_2",
        "cmp3": "Accepte_Campagne_3",
        "cmp4": "Accepte_Campagne_4",
        "cmp5": "Accepte_Campagne_5",
    },
]

def pick_schema(df: pd.DataFrame):
    """Choisit automatiquement le mapping le plus compatible avec le DF."""
    best = None
    best_score = -1
    for m in COLMAPS:
        keys = list(m.values())
        score = sum(1 for k in keys if k in df.columns)
        if score > best_score:
            best = m
            best_score = score
    return best

# ------------------ core ------------------

def engineer_features(df: pd.DataFrame, ref_year: int = 2014) -> pd.DataFrame:
    m = pick_schema(df)
    if m is None:
        raise ValueError("Impossible de détecter le schéma de colonnes (FR/EN).")

    # colonnes nécessaires minimum
    required = [
        m["vin"], m["fruits"], m["viande"], m["poisson"], m["sucreries"], m["luxe"],
        m["achats_web"], m["achats_catalogue"], m["achats_magasin"],
        m["visites_web"], m["enfant"], m["ado"], m["annee_naissance"], m["revenu"],
        m["cmp1"], m["cmp2"], m["cmp3"], m["cmp4"], m["cmp5"]
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Colonnes manquantes : {missing}")

    # force numériques
    num_cols = [
        m["vin"], m["fruits"], m["viande"], m["poisson"], m["sucreries"], m["luxe"],
        m["achats_web"], m["achats_catalogue"], m["achats_magasin"], m["visites_web"],
        m["enfant"], m["ado"], m["annee_naissance"], m["revenu"],
        m["cmp1"], m["cmp2"], m["cmp3"], m["cmp4"], m["cmp5"],
    ]
    df = ensure_numeric(df, num_cols)

    # 1) Dépense_totale
    df["Depense_totale"] = (
        df[m["vin"]]
        + df[m["fruits"]]
        + df[m["viande"]]
        + df[m["poisson"]]
        + df[m["sucreries"]]
        + df[m["luxe"]]
    )

    # 2) Depense_plaisir
    df["Depense_plaisir"] = df[m["vin"]] + df[m["sucreries"]] + df[m["luxe"]]

    # 3) Part_plaisir
    df["Part_plaisir"] = df["Depense_plaisir"] / df["Depense_totale"].replace(0, np.nan)

    # 4) Taux_depense_sur_revenu
    df["Taux_depense_sur_revenu"] = df["Depense_totale"] / df[m["revenu"]].replace(0, np.nan)

    # 5) Total_achats
    df["Total_achats"] = df[m["achats_web"]] + df[m["achats_catalogue"]] + df[m["achats_magasin"]]

    # 6) Part_achats_en_ligne
    df["Part_achats_en_ligne"] = df[m["achats_web"]] / df["Total_achats"].replace(0, np.nan)

    # 7) Part_achats_catalogue
    df["Part_achats_catalogue"] = df[m["achats_catalogue"]] / df["Total_achats"].replace(0, np.nan)

    # 8) Taux_visite_achat_web
    df["Taux_visite_achat_web"] = df[m["achats_web"]] / df[m["visites_web"]].replace(0, np.nan)

    # 9) Nb_enfants_total
    df["Nb_enfants_total"] = df[m["enfant"]] + df[m["ado"]]

    # 10) Age
    df["Age"] = ref_year - df[m["annee_naissance"]]

    # 11) Score_engagement_marketing
    df["Score_engagement_marketing"] = (
        df[m["cmp1"]] + df[m["cmp2"]] + df[m["cmp3"]] + df[m["cmp4"]] + df[m["cmp5"]]
    )

    # 12) Taux_acceptation_campagne
    df["Taux_acceptation_campagne"] = df["Score_engagement_marketing"] / 5.0

    return df

# ------------------ CLI ------------------

def main():
    p = argparse.ArgumentParser(description="Création d'indicateurs (feature engineering)")
    p.add_argument("--input", required=True, help="Chemin du CSV d'entrée")
    p.add_argument("--output", required=True, help="Chemin du CSV enrichi à écrire")
    p.add_argument("--ref-year", type=int, default=2014, help="Année de référence pour le calcul de l'âge (défaut 2014)")
    args = p.parse_args()

    try:
        df = pd.read_csv(args.input)
    except Exception as e:
        print(f"[ERREUR] Lecture CSV: {e}")
        sys.exit(1)

    try:
        df_out = engineer_features(df, ref_year=args.ref_year)
    except Exception as e:
        print(f"[ERREUR] Calcul des features: {e}")
        sys.exit(2)

    try:
        df_out.to_csv(args.output, index=False)
    except Exception as e:
        print(f"[ERREUR] Écriture CSV: {e}")
        sys.exit(3)

    # Rendu terminal concis
    n_rows, n_cols = df_out.shape
    created_cols = [
        "Depense_totale","Depense_plaisir","Part_plaisir","Taux_depense_sur_revenu",
        "Total_achats","Part_achats_en_ligne","Part_achats_catalogue","Taux_visite_achat_web",
        "Nb_enfants_total","Age","Score_engagement_marketing","Taux_acceptation_campagne"
    ]

    print("\n" + "="*72)
    print(" FEATURES CRÉÉES ".center(72, "="))
    print("="*72)
    for c in created_cols:
        miss = df_out[c].isna().mean()
        print(f" - {c:28s} | NaN: {miss:6.2%} | exemple: {df_out[c].dropna().head(1).to_list()[0] if df_out[c].notna().any() else '—'}")
    print("-"*72)
    print(f"Lignes: {n_rows:,}  |  Colonnes: {n_cols:,}".replace(",", " "))
    print(f"Fichier écrit -> {args.output}")
    print("="*72 + "\n")

if __name__ == "__main__":
    main()


# Indicateurs clés de performance (KPIs Marketing)

Cette section calcule et affiche les indicateurs marketing globaux et par campagne à partir du jeu de données client nettoyé.  

- Fournir une vue synthétique de la performance client et marketing.  
- Mesurer les résultats des campagnes 1 à 5 (taux d’acceptation, CA, fréquence, AOV, etc.).  
- Évaluer la qualité et la fiabilité des données utilisées.  
- Identifier les clients à fort potentiel (CLV) et ceux à risque selon leur récence d’achat.  
- Générer une synthèse actionnable pour orienter les décisions marketing.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
kpi_terminal.py — KPIs marketing (FR) avec rendu terminal soigné, SANS groupe test/contrôle.
Ajoute l'analyse des campagnes 1..5 : taux d'acceptation, CA des acceptants, etc.

Usage (exemples) :
  python kpi_terminal.py --file data/camp_market_clean.csv --cout-campagne 12000 --marge 0.30 --horizon 2
  python kpi_terminal.py --file data/camp_market_clean.csv --width 110
"""

import argparse
from datetime import datetime
import math
import sys
import numpy as np
import pandas as pd

# ---------- Helpers d'affichage (ASCII/Unicode, aucune dépendance externe) ----------

USE_UNICODE = sys.stdout.encoding and "UTF" in sys.stdout.encoding.upper()

def _hline(width, heavy=False):
    if USE_UNICODE:
        h = "═" if heavy else "─"
        return h * width
    return "-" * width

def _box(title, lines, width=80):
    title = f" {title.strip()} "
    if USE_UNICODE:
        tl, tr, bl, br = "╔", "╗", "╚", "╝"
        top = tl + _hline(width - 2, True) + tr
        bot = bl + _hline(width - 2, True) + br
        title_bar = "╠" + title.center(width - 2, "═") + "╣"
        content = []
        for ln in lines:
            s = ("" if ln is None else str(ln))[:width - 4]
            content.append("║ " + s.ljust(width - 4) + " ║")
        return "\n".join([top, title_bar] + content + [bot])
    else:
        top = "+" + _hline(width - 2) + "+"
        title_bar = "+" + title.center(width - 2, "-") + "+"
        bot = "+" + _hline(width - 2) + "+"
        content = []
        for ln in lines:
            s = ("" if ln is None else str(ln))[:width - 4]
            content.append("| " + s.ljust(width - 4) + " |")
        return "\n".join([top, title_bar] + content + [bot])

def fmt_num(x, decimals=2):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        if isinstance(x, (int, np.integer)):
            return f"{int(x):,}".replace(",", " ")
        return f"{float(x):,.{decimals}f}".replace(",", " ")
    except Exception:
        return "-"

def fmt_pct(x, decimals=2):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"{100*float(x):.{decimals}f}%"
    except Exception:
        return "-"

def safe_div(a, b):
    try:
        if b == 0 or b is None or (isinstance(b, float) and math.isnan(b)):
            return float("nan")
        return a / b
    except Exception:
        return float("nan")

def print_kv_table(title, rows, width=80, key_w=34):
    lines = []
    for k, v in rows:
        k = str(k); v = str(v)
        if len(k) > key_w:
            k = k[:key_w-1] + "…"
        pad = " " * max(1, key_w - len(k))
        lines.append(f"{k}{pad} : {v}")
    print(_box(title, lines, width=width))
    print()

def print_table(title, headers, rows, width=80, col_w=None):
    if col_w is None:
        n = len(headers)
        col_w = [max(len(str(h)), 10) for h in headers]
        for r in rows:
            for i, c in enumerate(r):
                col_w[i] = max(col_w[i], len(str(c)))
        total = sum(col_w) + 3*(len(headers)-1)
        target = min(max(total, 40), width-4)
        if total > target:
            ratio = target/total
            col_w = [max(8, int(w*ratio)) for w in col_w]

    def fmt_row(cells):
        out = []
        for i, c in enumerate(cells):
            s = str(c)
            if len(s) > col_w[i]:
                s = s[:col_w[i]-1] + "…"
            out.append(s.ljust(col_w[i]))
        return " | ".join(out)

    lines = []
    lines.append(fmt_row(headers))
    lines.append("-" * len(lines[0]))
    for r in rows:
        lines.append(fmt_row(r))
    print(_box(title, lines, width=width))
    print()

# ---------- Nettoyage & conversions ----------

def ensure_numeric(df, cols):
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c].astype(str)
        s = s.str.replace(",", ".", regex=False)
        s = s.str.replace(r'[^0-9\.\-eE+]', '', regex=True)
        df[c] = pd.to_numeric(s, errors="coerce")
    return df

# ---------- Indicateurs de qualité & fiabilité ----------

def iqr_outlier_mask(series):
    q1, q3 = np.nanpercentile(series, [25, 75])
    iqr = q3 - q1
    low = q1 - 1.5*iqr
    high = q3 + 1.5*iqr
    return (series < low) | (series > high)

def reliability_score(metrics):
    score = 100.0
    score -= max(0, (1 - metrics["coverage"])) * 30.0
    score -= min(30.0, metrics["bad_dates_pct"] * 100 * 0.5)
    score -= min(20.0, (metrics["dup_ids"] > 0) * 10.0 + min(metrics["dup_ids"], 10) * 1.0)
    score -= min(10.0, metrics["zero_freq_pct"] * 100 * 0.3)
    score -= min(20.0, metrics["outlier_pct"] * 100 * 0.5)
    return max(0.0, min(100.0, score))

# ---------- KPI principaux (sans groupe) + Campagnes ----------

def build_kpis(df, cout_campagne=0.0, marge=0.30, horizon=2.0):
    required = [
        "Identifiant","Année_naissance","Education","Situation_matrimoniale","Revenu",
        "Enfant_charge","Ado_charge","Date_acquisition_client","Nombre_jours_depuis_dernier_achat",
        "Montant_vin","Montant_fruits","Montant_viande","Montant_poisson","Montant_sucreries","Montant_luxe",
        "Nb_achats_promo","Nb_achats_en_ligne","Achats_catalogue","Achats_magasin","Nb_visites_web_mois",
        "Accepte_Campagne_3","Accepte_Campagne_4","Accepte_Campagne_5","Accepte_Campagne_1","Accepte_Campagne_2",
        "Reclamation_client","Z_CostContact","Z_Revenue","Response"
    ]
    present = [c for c in required if c in df.columns]
    coverage = len(present) / len(required)

    if "Date_acquisition_client" in df.columns and not pd.api.types.is_datetime64_any_dtype(df["Date_acquisition_client"]):
        df["Date_acquisition_client"] = pd.to_datetime(df["Date_acquisition_client"], errors="coerce")

    numeric_cols = [
        "Identifiant","Année_naissance","Revenu","Enfant_charge","Ado_charge",
        "Nombre_jours_depuis_dernier_achat","Nb_achats_promo","Nb_achats_en_ligne",
        "Achats_catalogue","Achats_magasin","Nb_visites_web_mois",
        "Z_CostContact","Z_Revenue","Response",
        "Montant_vin","Montant_fruits","Montant_viande","Montant_poisson","Montant_sucreries","Montant_luxe",
        "Accepte_Campagne_1","Accepte_Campagne_2","Accepte_Campagne_3","Accepte_Campagne_4","Accepte_Campagne_5",
    ]
    df = ensure_numeric(df, numeric_cols)

    if "Response" in df.columns:
        df["Response"] = df["Response"].fillna(0).astype(float)

    today = pd.Timestamp.today().normalize()
    if "Année_naissance" in df.columns:
        df["Age"] = (today.year - df["Année_naissance"]).astype(float)
    if "Date_acquisition_client" in df.columns:
        df["Tenure_j"] = (today - df["Date_acquisition_client"]).dt.days.astype(float)
    if "Nombre_jours_depuis_dernier_achat" in df.columns:
        df["Recency_j"] = df["Nombre_jours_depuis_dernier_achat"].astype(float)

    spend_cols = [c for c in [
        "Montant_vin","Montant_fruits","Montant_viande",
        "Montant_poisson","Montant_sucreries","Montant_luxe"
    ] if c in df.columns]
    df["Spend_total"] = df[spend_cols].sum(axis=1)

    purch_cols = [c for c in ["Achats_catalogue","Achats_magasin","Nb_achats_en_ligne"] if c in df.columns]
    df["Freq"] = df[purch_cols].sum(axis=1)
    df["AOV"] = df["Spend_total"] / df["Freq"].replace(0, np.nan)

    df["Part_online"]    = df["Nb_achats_en_ligne"] / df["Freq"].replace(0, np.nan)
    df["Part_magasin"]   = df["Achats_magasin"] / df["Freq"].replace(0, np.nan)
    df["Part_catalogue"] = df["Achats_catalogue"] / df["Freq"].replace(0, np.nan)
    for cat in ["vin","fruits","viande","poisson","sucreries","luxe"]:
        col = f"Montant_{cat}"
        if col in df.columns:
            df[f"Part_{cat}"] = df[col] / df["Spend_total"].replace(0, np.nan)

    if spend_cols:
        df["Cross_sell"] = (df[spend_cols] > 0).sum(axis=1) / len(spend_cols)
    else:
        df["Cross_sell"] = np.nan
    df["Promo_ratio"] = df["Nb_achats_promo"] / df["Freq"].replace(0, np.nan)

    df["Charge_index"] = df["Enfant_charge"] + df["Ado_charge"]
    df["Has_claim"] = (df["Reclamation_client"] > 0).astype(int)

    df["Tenure_annees"] = (df.get("Tenure_j", pd.Series(np.nan, index=df.index)) / 365).clip(lower=1/12)
    df["Freq_par_an"] = df["Freq"] / df["Tenure_annees"]
    df["CLV_lite"] = df["AOV"] * df["Freq_par_an"] * float(marge) * float(horizon)

    # ---------- Qualité & Fiabilité ----------
    n_total = len(df)
    bad_dates_pct = float(df["Date_acquisition_client"].isna().mean()) if "Date_acquisition_client" in df.columns else 1.0
    dup_ids = int(df["Identifiant"].duplicated().sum()) if "Identifiant" in df.columns else 0
    zero_freq_pct = float((df["Freq"].fillna(0) == 0).mean()) if "Freq" in df.columns else 1.0
    outlier_mask = iqr_outlier_mask(df["Spend_total"].fillna(0).astype(float))
    outlier_pct = float(outlier_mask.mean()) if n_total > 0 else 0.0

    rel_inputs = {
        "coverage": coverage,
        "bad_dates_pct": bad_dates_pct,
        "dup_ids": dup_ids,
        "zero_freq_pct": zero_freq_pct,
        "outlier_pct": outlier_pct
    }
    score_rel = reliability_score(rel_inputs)
    label_rel = ("Excellente" if score_rel >= 90 else
                 "Bonne" if score_rel >= 75 else
                 "Moyenne" if score_rel >= 60 else
                 "Faible")

    # ---------- KPI Globaux ----------
    global_kpi = {
        "Clients (total)": n_total,
        "Taux de réponse (global)": float(df["Response"].mean()) if "Response" in df.columns else float("nan"),
        "Freq moyenne": float(df["Freq"].mean()),
        "AOV moyen": float(df["AOV"].mean()),
        "Dépense totale moyenne": float(df["Spend_total"].mean()),
        "CLV lite moyenne": float(df["CLV_lite"].mean()),
        "Promo_ratio moyen": float(df["Promo_ratio"].mean()),
        "Part online moyenne": float(df["Part_online"].mean()),
        "Réclamants (%)": float(df["Has_claim"].mean()),
    }

    # ---------- IC95% pour le taux de réponse global (Wilson) ----------
    def wilson_ci(p, n, z=1.96):
        if n == 0 or p != p:
            return (float("nan"), float("nan"))
        denom = 1 + z**2 / n
        center = (p + z**2/(2*n)) / denom
        margin = z * math.sqrt((p*(1-p)/n) + (z**2/(4*n**2))) / denom
        return (max(0.0, center - margin), min(1.0, center + margin))

    p = global_kpi["Taux de réponse (global)"]
    n = n_total
    ci_low, ci_high = wilson_ci(p, n)

    # ---------- Campagnes 1..5 ----------
    camp_cols = [c for c in ["Accepte_Campagne_1","Accepte_Campagne_2","Accepte_Campagne_3","Accepte_Campagne_4","Accepte_Campagne_5"] if c in df.columns]
    camp_rows = []
    for c in camp_cols:
        k = int(c.split("_")[-1])
        acc = df[c].fillna(0).astype(int)
        n_accept = int(acc.sum())
        rate = float(acc.mean()) if len(acc) else float("nan")

        mask = acc == 1
        ca_total_accept = float(df.loc[mask, "Spend_total"].sum())
        ca_moy_accept = float(df.loc[mask, "Spend_total"].mean()) if n_accept > 0 else float("nan")
        ca_moy_par_client_global = safe_div(ca_total_accept, n_total)

        freq_acc = float(df.loc[mask, "Freq"].mean()) if n_accept > 0 else float("nan")
        aov_acc  = float(df.loc[mask, "AOV"].mean())  if n_accept > 0 else float("nan")
        resp_acc = float(df.loc[mask, "Response"].mean()) if "Response" in df.columns and n_accept > 0 else float("nan")

        camp_rows.append({
            "Campagne": k,
            "Taux_acceptation": rate,
            "Acceptants": n_accept,
            "CA_total_acceptants": ca_total_accept,
            "CA_moyen_acceptant": ca_moy_accept,
            "CA_moyen_par_client_global": ca_moy_par_client_global,
            "Freq_moy_acc": freq_acc,
            "AOV_moy_acc": aov_acc,
            "Taux_resp_des_acceptants": resp_acc
        })

    camp_df = pd.DataFrame(camp_rows).sort_values(["Taux_acceptation","CA_total_acceptants"], ascending=[False, False]) if camp_rows else pd.DataFrame()

    # ---------- Segments ----------
    df["CLV_lite"] = pd.to_numeric(df["CLV_lite"], errors="coerce").fillna(0.0)
    top_val = df.nlargest(5, "CLV_lite")[["Identifiant","CLV_lite","Spend_total","Freq","AOV","Part_online","Promo_ratio"]]
    risques = df.sort_values("Recency_j", ascending=False).head(5)[["Identifiant","Recency_j","Spend_total","Freq","Part_online"]]

    quality = {
        "coverage": coverage,
        "bad_dates_pct": bad_dates_pct,
        "dup_ids": dup_ids,
        "zero_freq_pct": zero_freq_pct,
        "outlier_pct": outlier_pct,
        "score": score_rel,
        "label": label_rel
    }
    ci = {"p": p, "n": n, "low": ci_low, "high": ci_high}
    return df, global_kpi, quality, ci, top_val, risques, camp_df

# ---------- Main ----------

def main():
    parser = argparse.ArgumentParser(description="KPIs marketing – rendu terminal soigné (sans groupes) + analyse campagnes")
    parser.add_argument("--file", required=True, help="Chemin du CSV nettoyé")
    parser.add_argument("--cout-campagne", type=float, default=0.0, help="Coût total de la campagne (monétaire)")
    parser.add_argument("--marge", type=float, default=0.30, help="Marge (%) utilisée pour CLV lite (ex: 0.30)")
    parser.add_argument("--horizon", type=float, default=2.0, help="Horizon (années) pour CLV lite (ex: 2)")
    parser.add_argument("--width", type=int, default=100, help="Largeur d'affichage")
    args = parser.parse_args()

    try:
        df = pd.read_csv(args.file, parse_dates=["Date_acquisition_client"])
    except Exception as e:
        print(f"Erreur de lecture CSV: {e}")
        sys.exit(1)

    try:
        df_out, global_kpi, quality, ci, top_val, risques, camp_df = build_kpis(
            df,
            cout_campagne=args.cout_campagne,
            marge=args.marge,
            horizon=args.horizon
        )
    except Exception as e:
        print(f"Erreur de calcul KPI: {e}")
        sys.exit(2)

    width = args.width

    header_lines = [
        f"Fichier : {args.file}",
        f"Date exécution : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Paramètres : coût={fmt_num(args.cout_campagne)} | marge={fmt_pct(args.marge)} | horizon={fmt_num(args.horizon,1)} an(s)"
    ]
    print(_box("TABLEAU DE BORD KPI – TERMINAL (GLOBAL + CAMPAGNES)", header_lines, width=width))
    print()

    qual_lines = [
        f"Couverture colonnes clés           : {fmt_pct(quality['coverage'])}",
        f"Dates invalides                    : {fmt_pct(quality['bad_dates_pct'])}",
        f"Identifiants dupliqués (#)         : {fmt_num(quality['dup_ids'], 0)}",
        f"Freq = 0 (impact AOV/parts)        : {fmt_pct(quality['zero_freq_pct'])}",
        f"Outliers dépense totale            : {fmt_pct(quality['outlier_pct'])}",
        f"Score de fiabilité (0-100)         : {fmt_num(quality['score'], 0)}  ({quality['label']})",
    ]
    print(_box("QUALITÉ DES DONNÉES & FIABILITÉ", qual_lines, width=width))
    print()

    rows_global = [
        ("Clients (total)", fmt_num(global_kpi["Clients (total)"], 0)),
        ("Taux de réponse (global)", fmt_pct(global_kpi["Taux de réponse (global)"])),
        ("Freq moyenne", fmt_num(global_kpi["Freq moyenne"], 2)),
        ("AOV moyen", fmt_num(global_kpi["AOV moyen"], 2)),
        ("Dépense totale moyenne", fmt_num(global_kpi["Dépense totale moyenne"], 2)),
        ("CLV lite moyenne", fmt_num(global_kpi["CLV lite moyenne"], 2)),
        ("Promo_ratio moyen", fmt_pct(global_kpi["Promo_ratio moyen"])),
        ("Part online moyenne", fmt_pct(global_kpi["Part online moyenne"])),
        ("Réclamants (%)", fmt_pct(global_kpi["Réclamants (%)"])),
    ]
    print_kv_table("KPI GLOBAUX (SANS TRT/CTL)", rows_global, width=width)

    ic_lines = [
        f"Taux de réponse (global) : {fmt_pct(ci['p'])}  (IC95% {fmt_pct(ci['low'])} ; {fmt_pct(ci['high'])})",
        f"Taille d'échantillon (n) : {fmt_num(ci['n'], 0)}",
    ]
    print(_box("CONFIANCE STATISTIQUE (IC 95%) — TAUX GLOBAL", ic_lines, width=width))
    print()

    if not camp_df.empty:
        headers_c = [
            "Campagne","Taux_acceptation","Acceptants",
            "CA_total_acceptants","CA_moyen_acceptant","CA_moyen_par_client",
            "Freq_moy_acc","AOV_moy_acc","Taux_resp_acc"
        ]
        rows_c = []
        for _, r in camp_df.iterrows():
            rows_c.append([
                int(r.Campagne),
                fmt_pct(r.Taux_acceptation),
                fmt_num(r.Acceptants, 0),
                fmt_num(r.CA_total_acceptants, 2),
                fmt_num(r.CA_moyen_acceptant, 2),
                fmt_num(r.CA_moyen_par_client_global, 2),
                fmt_num(r.Freq_moy_acc, 2),
                fmt_num(r.AOV_moy_acc, 2),
                fmt_pct(r.Taux_resp_des_acceptants),
            ])
        print_table("KPI PAR CAMPAGNE (HISTORIQUE)", headers_c, rows_c, width=width)

        best_rate = camp_df.sort_values("Taux_acceptation", ascending=False).iloc[0]
        best_ca   = camp_df.sort_values("CA_total_acceptants", ascending=False).iloc[0]
        best_cam_lines = [
            f"• Plus haut taux d’acceptation : Campagne {int(best_rate.Campagne)} → {fmt_pct(best_rate.Taux_acceptation)} (acceptants={fmt_num(best_rate.Acceptants,0)})",
            f"• Plus gros CA des acceptants : Campagne {int(best_ca.Campagne)} → {fmt_num(best_ca.CA_total_acceptants,2)}",
            f"• CA moyen/acceptant le + élevé : Campagne {int(camp_df.iloc[camp_df['CA_moyen_acceptant'].idxmax()].Campagne)} "
            f"→ {fmt_num(camp_df['CA_moyen_acceptant'].max(),2)}",
        ]
        print(_box("SYNTHÈSE — MEILLEURES CAMPAGNES", best_cam_lines, width=width))
        print()
    else:
        print(_box("KPI PAR CAMPAGNE (HISTORIQUE)", ["Colonnes Accepte_Campagne_1..5 absentes."], width=width))
        print()

    headers_top = ["Identifiant","CLV_lite","Spend","Freq","AOV","Part_online","Promo_ratio"]
    rows_top = [
        [
            "-" if pd.isna(r.Identifiant) else int(r.Identifiant),
            fmt_num(r.CLV_lite,2),
            fmt_num(r.Spend_total,2),
            fmt_num(r.Freq,2),
            fmt_num(r.AOV,2),
            fmt_pct(r.Part_online),
            fmt_pct(r.Promo_ratio)
        ]
        for _, r in top_val.iterrows()
    ]
    print_table("TOP 5 CLIENTS PAR CLV (LITE)", headers_top, rows_top, width=width)

    headers_risk = ["Identifiant","Recency_j","Spend","Freq","Part_online"]
    rows_risk = [
        [
            "-" if pd.isna(r.Identifiant) else int(r.Identifiant),
            fmt_num(r.Recency_j,0),
            fmt_num(r.Spend_total,2),
            fmt_num(r.Freq,2),
            fmt_pct(r.Part_online)
        ]
        for _, r in risques.iterrows()
    ]
    print_table("TOP 5 À RISQUE (RECENCY ÉLEVÉE)", headers_risk, rows_risk, width=width)

    synth = [
        "• Accélérer la meilleure campagne (haut taux ou haut CA) et répliquer ses leviers.",
        "• Tester des optimisations sur les campagnes faibles (créa, ciblage, canal).",
        "• Activer upsell/cross-sell sur TOP CLV ; relancer clients à recency élevée avec offres non-promo."
    ]
    print(_box("SYNTHÈSE ACTIONNABLE (RÉSUMÉ)", synth, width=width))

if __name__ == "__main__":
    main()